<a href="https://colab.research.google.com/github/Rayhwcakep/PCVK/blob/main/week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import cv2 as cv
from google.colab.patches import cv2_imshow #for image display
from skimage import io
from skimage import transform
from PIL import Image
import matplotlib.pylab as plt

print(' Mengubah tingkat kecerahan citra ')
print('--------------------------------')
try:
    brightness = int(input('Masukkan nilai kecerahan: '))
except ValueError:
    print('Error, not a number')

#sesuaikan dengan file dan folder di drive Anda
original = cv.imread('/content/drive/MyDrive/Colab Notebooks/ptmalam.jpeg')
brightness_image = np.zeros(original.shape, original.dtype)

#akses per piksel
#np.clip digunakan untuk melakukan truncate pixel (nilai dibatasi antara 0-255)
for y in range(original.shape[0]):
    for x in range(original.shape[1]):
        for c in range(original.shape[2]):
            brightness_image[y,x,c] = np.clip(original[y,x,c] + brightness, 0, 255)

#cara simple tanpa for loop
brightness_image = cv.convertScaleAbs(original, beta=brightness)

final_frame = cv.hconcat((original, brightness_image))
cv2_imshow(final_frame)

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

# Baca citra asli (contoh: peppers.jpg)
original = cv.imread('/content/drive/MyDrive/Colab Notebooks/ptmalam.jpeg')

# Operasi inverse citra
inverse_image = 255 - original

# Gabungkan citra asli dan hasil inverse secara horizontal
final_frame = cv.hconcat((original, inverse_image))
cv2_imshow(final_frame)

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

print(' Mengubah kontras dan tingkat kecerahan citra ')
print('---------------------------------------------')
try:
    brightness = int(input('Masukkan tingkat kecerahan: '))
    contrast = float(input('Masukkan kontras: '))
except ValueError:
    print('Error, nilai harus berupa angka')

original = cv.imread('/content/drive/MyDrive/Colab Notebooks/ptmalam.jpeg')

# Hitung contrast factor F
factor = (259 * (contrast + 255)) / (255 * (259 - contrast))

# Konversi tipe data ke int agar tidak overflow saat operasi matematika
enhanced_image = original.astype(np.float32)

# Terapkan rumus kontras dan brightness per piksel menggunakan np.clip
enhanced_image = factor * (enhanced_image - 128) + 128 + brightness
contrast_image = np.clip(enhanced_image, 0, 255).astype(np.uint8)

# Tampilkan citra berdampingan
final_frame = cv.hconcat((original, contrast_image))
cv2_imshow(final_frame)

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

print(' Mengubah tingkat kecerahan citra dengan Transformasi Log ')
print('---------------------------------------------------------')
try:
    c_val = float(input('Masukkan nilai kecerahan '))
except ValueError:
    c_val = 30 # Nilai default jika input error

original = cv.imread('/content/drive/MyDrive/Colab Notebooks/ptmalam.jpeg')

# Ubah tipe data ke float
gray_float = original.astype(np.float32)

# Terapkan formula Logarithmic Transformation: s = c * log(1 + r)
log_image = c_val * np.log(1 + gray_float)

# Normalisasi kembali ke rentang 0-255 dan ubah ke uint8
log_image = cv.normalize(log_image, None, 0, 255, cv.NORM_MINMAX)
log_image = np.uint8(log_image)

final_frame = cv.hconcat((original, log_image))
cv2_imshow(final_frame)

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

original = cv.imread('/content/drive/MyDrive/Colab Notebooks/ptmalam.jpeg')
b, g, r = cv.split(original)

# a. Metode Averaging
gray_avg = ((r.astype(float) + g.astype(float) + b.astype(float)) / 3).astype(np.uint8)
gray_avg_bgr = cv.cvtColor(gray_avg, cv.COLOR_GRAY2BGR)

# b. Metode Lightness
max_rgb = np.maximum(np.maximum(r, g), b).astype(float)
min_rgb = np.minimum(np.minimum(r, g), b).astype(float)
gray_light = ((max_rgb + min_rgb) / 2).astype(np.uint8)
gray_light_bgr = cv.cvtColor(gray_light, cv.COLOR_GRAY2BGR)

# c. Metode Luminance (Luminosity)
gray_lumi = (0.21 * r.astype(float) + 0.72 * g.astype(float) + 0.07 * b.astype(float)).astype(np.uint8)
gray_lumi_bgr = cv.cvtColor(gray_lumi, cv.COLOR_GRAY2BGR)

# Tampilkan hasil (Contoh untuk Averaging, ganti variabel sesuai kebutuhan)
cv2_imshow(cv.hconcat((original, gray_avg_bgr)))
# cv2_imshow(cv.hconcat((original, gray_light_bgr)))
# cv2_imshow(cv.hconcat((original, gray_lumi_bgr)))

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

original = cv.imread('/content/drive/MyDrive/Colab Notebooks/ptmalam.jpeg')

# Konversi ke ruang warna HSV untuk mendeteksi warna merah dengan lebih akurat
hsv = cv.cvtColor(original, cv.COLOR_BGR2HSV)

# Definisikan rentang warna merah pada HSV
lower_red1 = np.array([0, 50, 50])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 50, 50])
upper_red2 = np.array([180, 255, 255])

mask1 = cv.inRange(hsv, lower_red1, upper_red1)
mask2 = cv.inRange(hsv, lower_red2, upper_red2)
mask = mask1 | mask2

# Ubah citra asli ke grayscale
gray = cv.cvtColor(original, cv.COLOR_BGR2GRAY)
gray_bgr = cv.cvtColor(gray, cv.COLOR_GRAY2BGR)

# Pisahkan bagian merah dan bagian non-merah (grayscale)
extracted_red = cv.bitwise_and(original, original, mask=mask)
inverse_mask = cv.bitwise_not(mask)
background_gray = cv.bitwise_and(gray_bgr, gray_bgr, mask=inverse_mask)

# Gabungkan kembali
final_output = cv.add(extracted_red, background_gray)
cv2_imshow(cv.hconcat((original, final_output)))

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

print(' Gamma Correction pada citra ')
print('----------------------------')
try:
    gamma = float(input('Masukkan nilai Gamma: '))
except ValueError:
    gamma = 0.5

original = cv.imread('/content/drive/MyDrive/Colab Notebooks/ptmalam.jpeg')

# Normalisasi piksel ke rentang [0, 1], pangkatkan dengan gamma, lalu kembalikan ke [0, 255]
normalized = original / 255.0
gamma_corrected = np.power(normalized, gamma) * 255.0
gamma_corrected = np.clip(gamma_corrected, 0, 255).astype(np.uint8)

final_frame = cv.hconcat((original, gamma_corrected))
cv2_imshow(final_frame)

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

bit_depth = int(input('Masukkan bit depth tujuan (1-8): '))
level = 255 / (np.pow(2, bit_depth) - 1) if hasattr(np, 'pow') else 255 / ((2 ** bit_depth) - 1)

original = cv.imread('/content/drive/MyDrive/Colab Notebooks/ptmalam.jpeg', cv.IMREAD_GRAYSCALE)

# Simulasi penurunan bit depth
depth_image = np.round(original / level) * level
depth_image = np.clip(depth_image, 0, 255).astype(np.uint8)

print(f"Bit depth awal: 8 bit")
print(f"Bit depth tujuan: {bit_depth} bit")
print(f"Jumlah level: {int(2 ** bit_depth)}")

final_frame = cv.hconcat((original, depth_image))
cv2_imshow(final_frame)

In [ ]:
import cv2 as cv
import numpy as np
import glob
from math import log10, sqrt
from google.colab.patches import cv2_imshow

# Fungsi untuk menghitung nilai PSNR
def calculate_psnr(img1, img2):
    mse = np.mean((img1.astype(float) - img2.astype(float)) ** 2)
    if mse == 0:
        return 100.0
    max_pixel = 255.0
    psnr = 20 * log10(max_pixel / sqrt(mse))
    return psnr

# Baca citra asli (ideal)
original_galaxy = cv.imread('/content/drive/MyDrive/Colab Notebooks/ptmalam.jpeg')

if original_galaxy is None:
    print("Error: Citra asli tidak ditemukan.")
else:
    # REVISI: Cukup gunakan string path teks biasa
    path_noise = '/content/drive/MyDrive/Colab Notebooks/noise.jpeg'

    cv_img = []
    # glob.glob akan memproses string path_noise menjadi list file
    for img_path in sorted(glob.glob(path_noise)):
        n = cv.imread(img_path)
        if n is not None:
            cv_img.append(n)

    jumlah_gambar_tersedia = len(cv_img)
    print(f"Total gambar ber-noise yang berhasil dibaca: {jumlah_gambar_tersedia}")

    # Cek apakah ada gambar yang berhasil dibaca
    if jumlah_gambar_tersedia == 0:
        print("Sistem tidak memproses karena tidak ada gambar noise yang ditemukan di path tersebut.")
    else:
        jumlah_list = [10, 20, 40, 80, 100]

        for jml in jumlah_list:
            if jumlah_gambar_tersedia >= jml:
                # Rata-ratakan sejumlah 'jml' citra pertama
                subset = cv_img[:jml]
                averaged_img = np.mean(subset, axis=0).astype(np.uint8)

                # Hitung PSNR terhadap citra asli
                psnr_val = calculate_psnr(original_galaxy, averaged_img)
                print(f"Jumlah Citra di-Average: {jml} | Nilai PSNR: {psnr_val:.2f} dB")

                # Tampilkan hasil
                cv2_imshow(averaged_img)
            else:
                print(f"Melewati {jml} rata-rata: Gambar tidak cukup (hanya punya {jumlah_gambar_tersedia}).")

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

# Pastikan nama file sesuai dengan foto terbaru yang di-upload
original = cv.imread('/content/drive/MyDrive/Colab Notebooks/WhatsApp Image 2025-05-17 at 20.31.21_ef99ac63.jpg')

if original is None:
    print("Error: Gambar tidak ditemukan. Periksa kembali path file.")
else:
    h, w = original.shape[:2]

    # Buat mask hitam
    mask = np.zeros((h, w), dtype=np.uint8)

    # --- PENYESUAIAN POSISI & UKURAN AGAR PAS DI WAJAH ---

    # Radius dikecilkan sedikit agar benar-benar pas di area wajah (sekitar 6.5% dari lebar gambar)
    radius_mask = int(w * 0.090)

    # 1. Masking wajah kiri (Pria baju hitam)
    # Digeser sedikit ke kiri (X = 24%) dan agak ke atas (Y = 44%)
    x_kiri = int(w * 0.30)
    y_kiri = int(h * 0.44)
    cv.circle(mask, (x_kiri, y_kiri), radius_mask, 255, -1)

    # 2. Masking wajah kanan (Pria berhelm putih)
    # Posisinya disesuaikan tepat di wajah/kaca helm (X = 68%, Y = 46%)
    x_kanan = int(w * 0.62)
    y_kanan = int(h * 0.47)
    # Jika ingin area helm juga tertutupi penuh, Anda bisa menambah radius di sini: (radius_mask + 10)
    cv.circle(mask, (x_kanan, y_kanan), radius_mask, 255, -1)

    # Gabungkan mask menjadi 3 channel (BGR)
    mask_bgr = cv.cvtColor(mask, cv.COLOR_GRAY2BGR)

    # Ambil bagian wajah saja (background sementara hitam)
    result_faces = cv.bitwise_and(original, mask_bgr)

    # --- MEMBUAT BACKGROUND MENJADI PUTIH ---
    result_white_bg = result_faces.copy()

    # Ubah semua piksel di luar area masking (mask == 0) menjadi putih
    result_white_bg[mask == 0] = [255, 255, 255]

    # Tampilkan hasil secara berdampingan
    print("Hasil Masking Wajah")
    cv2_imshow(cv.hconcat([original, mask_bgr, result_white_bg]))

tugas nomer 10
Metode yang Dipilih: Gamma Correction (dengan nilai $\gamma < 1$, misal $\gamma = 0.4$ atau $0.5$).  Alasan: Foto malam hari umumnya memiliki masalah underexposure (kurang cahaya) di mana detail gelap tersembunyi. Transformasi linear biasa (penambahan konstan) sering kali membuat bagian yang sudah terang menjadi terlalu silau (overexposed / washed out). Gamma correction dengan $\gamma < 1$ secara non-linear merentangkan intensitas warna pada area gelap secara lebih natural tanpa merusak detail terang secara berlebihan.Resiko: Dapat memunculkan noise digital (bintik-bintik/grain) yang tadinya tersembunyi di area gelap karena ikut terdongkrak tingkat kecerahannya.

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

# Pastikan nama file sesuai dengan foto terbaru yang di-upload
original = cv.imread('/content/drive/MyDrive/Colab Notebooks/WhatsApp Image 2025-05-17 at 20.31.21_ef99ac63.jpg')

if original is None:
    print("Error: Gambar tidak ditemukan. Periksa kembali path file.")
else:
    h, w = original.shape[:2]

    # Buat mask hitam
    mask = np.zeros((h, w), dtype=np.uint8)

    # --- PENYESUAIAN POSISI & UKURAN AGAR PAS DI WAJAH ---

    # Radius dikecilkan sedikit agar benar-benar pas di area wajah (sekitar 6.5% dari lebar gambar)
    radius_mask = int(w * 0.090)

    # 1. Masking wajah kiri (Pria baju hitam)
    # Digeser sedikit ke kiri (X = 24%) dan agak ke atas (Y = 44%)
    x_kiri = int(w * 0.30)
    y_kiri = int(h * 0.44)
    cv.circle(mask, (x_kiri, y_kiri), radius_mask, 255, -1)

    # 2. Masking wajah kanan (Pria berhelm putih)
    # Posisinya disesuaikan tepat di wajah/kaca helm (X = 68%, Y = 46%)
    x_kanan = int(w * 0.62)
    y_kanan = int(h * 0.47)
    # Jika ingin area helm juga tertutupi penuh, Anda bisa menambah radius di sini: (radius_mask + 10)
    cv.circle(mask, (x_kanan, y_kanan), radius_mask, 255, -1)

    # Gabungkan mask menjadi 3 channel (BGR)
    mask_bgr = cv.cvtColor(mask, cv.COLOR_GRAY2BGR)

    # Ambil bagian wajah saja (background sementara hitam)
    result_faces = cv.bitwise_and(original, mask_bgr)

    # --- MEMBUAT BACKGROUND MENJADI PUTIH ---
    result_white_bg = result_faces.copy()

    # Ubah semua piksel di luar area masking (mask == 0) menjadi putih
    result_white_bg[mask == 0] = [255, 255, 255]

    # Tampilkan hasil secara berdampingan
    print("Hasil Masking Wajah")
    cv2_imshow(cv.hconcat([original, mask_bgr, result_white_bg]))

In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

# 1. Baca Citra Input (Silakan gunakan path foto Anda yang sudah ada)
original = cv.imread('/content/drive/MyDrive/Colab Notebooks/WhatsApp Image 2025-05-17 at 20.31.21_ef99ac63.jpg')
h, w = original.shape[:2]

# 2. Buat Citra Mask (Background hitam, lingkaran putih di area wajah)
mask = np.zeros((h, w), dtype=np.uint8)
radius_mask = int(w * 0.090)
x_kiri, y_kiri = int(w * 0.30), int(h * 0.44)
x_kanan, y_kanan = int(w * 0.62), int(h * 0.47)
cv.circle(mask, (x_kiri, y_kiri), radius_mask, 255, -1)
cv.circle(mask, (x_kanan, y_kanan), radius_mask, 255, -1)
mask_bgr = cv.cvtColor(mask, cv.COLOR_GRAY2BGR)

# --- PROSES OPERASI LOGIKA ---

# 1. NOT (Komplemen) - Diterapkan pada citra asli
res_not = cv.bitwise_not(original)

# 2. OR
res_or = cv.bitwise_or(original, mask_bgr)

# 3. AND
res_and = cv.bitwise_and(original, mask_bgr)

# 4. NAND (Not AND) - Dibuat dengan melakukan operasi AND, lalu di-NOT kan
res_nand = cv.bitwise_not(res_and)

# 5. XOR
res_xor = cv.bitwise_xor(original, mask_bgr)

# --- MENAMPILKAN HASIL UNTUK SCREENSHOT ---
print("1. Hasil NOT:")
cv2_imshow(res_not)

print("2. Hasil OR:")
cv2_imshow(res_or)

print("3. Hasil AND:")
cv2_imshow(res_and)

print("4. Hasil NAND:")
cv2_imshow(res_nand)

print("5. Hasil XOR:")
cv2_imshow(res_xor)

# Catatan: Untuk kolom Image Input di tabel, Anda bisa screenshot variabel 'original' dan 'mask_bgr'

Operasi XOR menghasilkan nilai asli (tetap) jika bertemu dengan nilai 0 (Hitam), dan akan membalik (menginversi) nilai jika bertemu dengan nilai 1 (Putih / 255). Pada citra, latar belakang yang di-mask dengan warna hitam (0) akan mempertahankan gambar aslinya secara utuh, sedangkan area wajah yang di-mask dengan lingkaran putih (255) akan berubah warnanya menjadi negatif (inverse).

NAND adalah kebalikan dari hasil operasi AND. Setelah mendapatkan hasil irisan cropping dari operasi AND (wajah terlihat, latar hitam), fungsi NOT akan membalikkan nilai pikselnya secara total. Wajah yang tadinya memiliki warna asli akan berubah menjadi efek film negatif, sedangkan latar belakang yang tadinya hitam (0) akan berubah menjadi putih terang (255).

Operasi AND hanya akan menghasilkan nilai asli jika input dari mask bernilai 1 (Putih / 255). Karena area luar lingkaran pada citra mask berwarna hitam (0), maka operasi AND dengan nilai 0 akan menghasilkan hitam (0). Akibatnya, operasi ini bertindak sebagai pemotong/pemisah (cropping); hanya citra asli di dalam lingkaran (area putih) yang tetap terlihat, sedangkan area di luarnya menjadi hitam.

Operasi OR menghasilkan nilai 1 (Putih / 255) jika salah satu atau kedua input bernilai 1. Pada masking, area mask yang berwarna putih (255) akan "menang" dan menutupi citra asli menjadi putih pekat. Sedangkan area mask yang berwarna hitam (0) akan membiarkan nilai piksel citra asli tetap terlihat tanpa perubahan.

Operasi NOT membalikkan nilai biner setiap piksel pada citra input (dari 0 menjadi 1, dan 1 menjadi 0). Pada skala grayscale 0-255, nilai piksel dikurangkan dari 255 (misal: 255 menjadi 0, warna terang menjadi gelap). Akibatnya, citra output yang dihasilkan akan terlihat seperti efek film negatif atau inverse dari citra asli.


In [ ]:
import cv2 as cv
import numpy as np
from google.colab.patches import cv2_imshow

original = cv.imread('/content/drive/MyDrive/Colab Notebooks/WhatsApp Image 2025-05-17 at 20.31.21_ef99ac63.jpg')

# Langkah 1: Koreksi kontras
contrast = 1.4
factor = (259 * (contrast + 255)) / (255 * (259 - contrast))
adjusted = factor * (original.astype(np.float32) - 128) + 128
adjusted = np.clip(adjusted, 0, 255).astype(np.uint8)

# Langkah 2: Koreksi Gamma untuk pencahayaan
gamma = 0.7
normalized = adjusted / 255.0
final_output = np.power(normalized, gamma) * 255.0
final_output = np.clip(final_output, 0, 255).astype(np.uint8)

# Tampilkan Before (Kiri) dan After (Kanan)
cv2_imshow(cv.hconcat([original, final_output]))

Metode yang Dipilih: Kombinasi Contrast Correction Factor dan Gamma Correction.Alasan: Citra crayfish.jpg biasanya memiliki kondisi pencahayaan yang suram/berkabut (rendah kontras). Penggunaan penyesuaian kontras akan mempertegas batas objek udang dari latar belakangnya, sedangkan Gamma Correction membantu menyeimbangkan iluminasi keseluruhan.Parameter Terbaik: Kontras $C = 1.3$ hingga $1.5$ dan Gamma $\gamma = 0.7$.Kode Implementasi Before-After: